In [1]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from scipy.spatial import cKDTree
import numpy as np


In [3]:
path = "/home/thore/Downloads/2022_A_S.txt"
verkehr_df = pd.read_csv(path, sep=";", header=0, usecols=['Land', 'Strklas', 'Strnum', 'Datum', 'Wotag', 'Fahrtzw', 'Stunde', 'KFZ_R1','KFZ_R2', "Zst"]) #

In [4]:
verkehr_df.head()

,Zst,Land,Strklas,Strnum,Datum,Wotag,Fahrtzw,Stunde,KFZ_R1,KFZ_R2
0,8001,8,A,5,220101,6,s,1,201,185
1,8001,8,A,5,220101,6,s,2,450,339
2,8001,8,A,5,220101,6,s,3,334,387
3,8001,8,A,5,220101,6,s,4,228,282
4,8001,8,A,5,220101,6,s,5,157,214


In [23]:
#convert "Datum" to datetime (from int) (yymmdd)

timeseries = pd.to_datetime(verkehr_df["Datum"], format="%y%m%d", errors='coerce')
print(timeseries)
verkehr_df["Monat"] = timeseries.dt.month
verkehr_df.head()


0         2022-01-01
1         2022-01-01
2         2022-01-01
3         2022-01-01
4         2022-01-01
             ...    
7603675   2022-12-31
7603676   2022-12-31
7603677   2022-12-31
7603678   2022-12-31
7603679   2022-12-31
Name: Datum, Length: 7603680, dtype: datetime64[ns]


,Land,Strklas,Strnum,Datum,Wotag,Fahrtzw,Stunde,KFZ_R1,KFZ_R2,Monat
0,8,A,5,220101,6,s,1,201,185,1
1,8,A,5,220101,6,s,2,450,339,1
2,8,A,5,220101,6,s,3,334,387,1
3,8,A,5,220101,6,s,4,228,282,1
4,8,A,5,220101,6,s,5,157,214,1


In [24]:
verkehr_df.drop(columns=["Datum"], inplace=True)
verkehr_df.to_csv("verkehr.csv", sep=";", index=False)

In [44]:
plz_population = "/home/thore/Downloads/PLZ_Gebiete_509014527448523709.csv"
df_plz_pop = pd.read_csv(plz_population, sep=",", header=0,usecols=['Postleitzahl', 'Einwohner:Innen','Gebietgröße in km²'] ) #]

In [45]:
df_plz_pop.columns

Index(['Postleitzahl', 'Einwohner:Innen', 'Gebietgröße in km²'], dtype='object')

In [46]:
plz_df = df_plz_pop.copy()
plz_df['density'] = plz_df['Einwohner:Innen'] / plz_df['Gebietgröße in km²']

In [47]:
plz_df.columns

Index(['Postleitzahl', 'Einwohner:Innen', 'Gebietgröße in km²', 'density'], dtype='object')

In [48]:
plz_locality = "/home/thore/Downloads/plz_geocoord.csv"
df_plz_loc = pd.read_csv(plz_locality, sep=",", header=0)

In [49]:
df_plz_loc.columns

Index(['plz', 'lat', 'lng'], dtype='object')

In [50]:
# 2. Indizes setzen
plz_df = plz_df.set_index('Postleitzahl')
df_plz_loc = df_plz_loc.set_index('plz')

In [51]:


# 3. DataFrames zusammenführen
df = plz_df.join(df_plz_loc, how='inner')

# 4. Bevölkerungsdichte berechnen
df['Bevölkerungsdichte'] = df['Einwohner:Innen'] / df['Gebietgröße in km²']

# 5. GeoDataFrame erstellen
geometry = [Point(xy) for xy in zip(df['lng'], df['lat'])]
gdf = gpd.GeoDataFrame(df, geometry=geometry)
gdf.set_crs(epsg=4326, inplace=True)  # WGS84
gdf = gdf.to_crs(epsg=3857)  # Metrisches Koordinatensystem

# 6. Koordinaten für KD-Baum extrahieren
coords = np.array([(geom.x, geom.y) for geom in gdf.geometry])
tree = cKDTree(coords)

# 7. Bevölkerung im 100 km-Umkreis berechnen
radius = 100_000  # 100 km in Metern
indices = tree.query_ball_tree(tree, r=radius)

# 8. Gesamtbevölkerung im Umkreis berechnen
population_within_100km = []
for idx_list in indices:
    total_pop = gdf.iloc[idx_list]['Einwohner:Innen'].sum()
    population_within_100km.append(total_pop)

# 9. Ergebnisse zum GeoDataFrame hinzufügen
gdf['Einwohner im 100km-Umkreis'] = population_within_100km

# 10. Relevante Spalten auswählen
result = gdf[['Einwohner:Innen', 'Bevölkerungsdichte', 'Einwohner im 100km-Umkreis']]
result = result.reset_index()
result.head()

,index,Einwohner:Innen,Bevölkerungsdichte,Einwohner im 100km-Umkreis
0,64743,3,36.555943,5143069
1,35647,4855,108.571752,4929668
2,31195,5876,83.356428,2875037
3,31084,4892,91.921198,1685539
4,27639,17093,95.001191,864845


In [52]:
result.to_csv("plz_pop_data.csv", sep=";", index=False)